# 🎯 Stock Bot AutoEncoder Fine-tuning on Google Colab

이 노트북은 사전 훈련된 AutoEncoder를 특정 트레이딩 태스크에 맞게 Fine-tuning합니다.

## 📋 Overview
- **목적**: 사전 훈련된 임베딩을 트레이딩 태스크에 적응
- **태스크**: 분류, 회귀, 랭킹 지원
- **방법**: 낮은 학습률로 인코더 미세조정 + 새로운 태스크 헤드
- **훈련 시간**: 약 1-2시간 (기본 모델 대비 빠름)

## 🔧 Environment Setup

In [ ]:
# Install required packages
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install h5py numpy pandas tqdm matplotlib seaborn scikit-learn

# Check GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Setup directories
import os
os.chdir('/content')
!mkdir -p models logs data

# Data and model paths
DATA_PATH = '/content/drive/MyDrive/ColabData/models/stockbot/pre_training_data'
MODEL_PATH = '/content/drive/MyDrive/ColabData/models/stockbot/autoencoder_colab_20251011_154401'  # 사전 훈련된 모델

# Check paths
print(f"Data path exists: {os.path.exists(DATA_PATH)}")
print(f"Model path exists: {os.path.exists(MODEL_PATH)}")

if os.path.exists(MODEL_PATH):
    models = [d for d in os.listdir(MODEL_PATH) if os.path.isdir(os.path.join(MODEL_PATH, d))]
    print(f"Available models: {sorted(models)}")

In [ ]:
# GitHub 저장소 클론
from google.colab import userdata

try:
    token = userdata.get('GITHUB_TOKEN')
    use_token = True
    print("🔑 Using GitHub token")
except:
    print("⚠️ Using public clone")
    use_token = False

if not os.path.exists('/content/stock-bot2'):
    if use_token:
        !git clone https://{token}@github.com/gblue1223/stock-bot2.git /content/stock-bot2
    else:
        !git clone https://github.com/gblue1223/stock-bot2.git /content/stock-bot2
else:
    %cd /content/stock-bot2
    !git pull

%cd /content/stock-bot2
import sys
sys.path.append('/content/stock-bot2')

print("✅ Repository ready!")

## 📚 Import Libraries and Models

In [ ]:
# Standard libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import h5py
import json
import random
import math
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import time
from datetime import datetime
from sklearn.metrics import classification_report, confusion_matrix

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Import project modules
try:
    from ai_trader.embedding.autoencoder_model import MaskedAutoEncoder, AutoEncoderEmbedding
    from ai_trader.embedding.fine_tuning import (
        TradingTaskDataset, TradingTaskHead, FineTunedEmbedding, 
        FineTuner, load_pretrained_model
    )
    print("✅ Successfully imported fine-tuning modules")
except ImportError as e:
    print(f"❌ Failed to import: {e}")
    print("📝 Loading modules from files...")
    exec(open('/content/stock-bot2/ai_trader/embedding/autoencoder_model.py').read())
    exec(open('/content/stock-bot2/ai_trader/embedding/fine_tuning.py').read())

## 🎯 Pre-trained Model Selection

In [ ]:
# 사전 훈련된 모델 선택
print("🔍 Available pre-trained models:")

if os.path.exists(MODEL_PATH):
    model_dirs = [d for d in os.listdir(MODEL_PATH) if os.path.isdir(os.path.join(MODEL_PATH, d))]
    
    for i, model_dir in enumerate(sorted(model_dirs)):
        model_info_path = os.path.join(MODEL_PATH, model_dir, 'model_info.json')
        if os.path.exists(model_info_path):
            with open(model_info_path, 'r') as f:
                info = json.load(f)
                print(f"  {i}: {model_dir}")
                print(f"     - Val Loss: {info.get('best_val_loss', 'N/A')}")
                print(f"     - Embedding Dim: {info.get('config', {}).get('embedding_dim', 'N/A')}")
                print(f"     - Training Time: {info.get('training_time_minutes', 'N/A'):.1f}min")
        else:
            print(f"  {i}: {model_dir} (no info available)")
    
    # 최신 모델 자동 선택 (또는 수동 선택)
    if model_dirs:
        SELECTED_MODEL = sorted(model_dirs)[-1]  # 가장 최신 모델
        PRETRAINED_MODEL_PATH = os.path.join(MODEL_PATH, SELECTED_MODEL, 'model.pt')
        
        print(f"
🎯 Selected model: {SELECTED_MODEL}")
        print(f"📁 Model path: {PRETRAINED_MODEL_PATH}")
        
        # 수동 선택을 원하는 경우 아래 주석 해제
        # model_index = 0  # 원하는 모델 인덱스
        # SELECTED_MODEL = sorted(model_dirs)[model_index]
        # PRETRAINED_MODEL_PATH = os.path.join(MODEL_PATH, SELECTED_MODEL, 'model.pt')
    else:
        print("❌ No pre-trained models found!")
        PRETRAINED_MODEL_PATH = None
else:
    print("❌ Model directory not found!")
    PRETRAINED_MODEL_PATH = None